# 의미 기반 장기 메모리 검색

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `VectorStoreRetrieverMemory` 대신, 현재는 LangGraph Store에 장기 메모리를
JSON 문서로 저장하고 semantic index를 구성할 수 있습니다. 메모리는 시간순 전체
기록이 아니라 **현재 질문과 의미적으로 관련된 항목**을 찾는 데 사용합니다.

이 예제는 면접 대화를 turn 단위 문서로 저장하고 상위 1개 결과를 검색합니다.
`InMemoryStore`는 실습용이며 운영에서는 영속 store를 사용합니다.


In [1]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv numpy


In [2]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langgraph.store.base import IndexConfig
from langgraph.store.memory import InMemoryStore

load_dotenv()

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("EMBEDDING_DIMENSIONS", "1536"))

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
memory_store = InMemoryStore(
    index=IndexConfig(
        embed=embeddings,
        dims=EMBEDDING_DIMENSIONS,
        fields=["text"],
    )
)


## 메모리 문서 저장

namespace에 사용자 ID와 메모리 종류를 넣어 다른 사용자의 데이터와 섞이지 않게 합니다.
`fields=["text"]`로 설정했으므로 `text` 필드만 embedding됩니다.


In [3]:
namespace = ("users", "candidate-001", "interview-memories")

interview_turns = [
    (
        "자기소개 부탁드립니다.",
        "컴퓨터 과학을 전공한 신입 개발자이며 자바와 파이썬을 사용했습니다.",
    ),
    (
        "프로젝트에서 어떤 역할을 맡았나요?",
        "백엔드 개발자로서 서버 로직, REST API, 데이터베이스 설계를 담당했습니다.",
    ),
    (
        "팀 프로젝트의 어려움은 어떻게 해결했나요?",
        "정기 미팅으로 진행 상황을 공유하고 문제를 함께 논의했습니다.",
    ),
    (
        "개발자로서 강점은 무엇인가요?",
        "빠른 학습 능력, 문제 해결 능력, 협업을 강점으로 생각합니다.",
    ),
]

for index, (human, ai) in enumerate(interview_turns, start=1):
    memory_store.put(
        namespace,
        f"turn-{index}",
        {"text": f"면접관: {human}\n지원자: {ai}", "turn": index},
    )


In [4]:
query = "면접자의 전공과 사용해 본 프로그래밍 언어는 무엇인가요?"
results = memory_store.search(namespace, query=query, limit=1)

for item in results:
    print("score:", item.score)
    print(item.value["text"])


score: 0.4696176977705822
면접관: 자기소개 부탁드립니다.
지원자: 컴퓨터 과학을 전공한 신입 개발자이며 자바와 파이썬을 사용했습니다.


In [5]:
query = "지원자가 프로젝트에서 맡은 업무는 무엇인가요?"
results = memory_store.search(namespace, query=query, limit=1)

for item in results:
    print("score:", item.score)
    print(item.value["text"])


score: 0.544776868114125
면접관: 프로젝트에서 어떤 역할을 맡았나요?
지원자: 백엔드 개발자로서 서버 로직, REST API, 데이터베이스 설계를 담당했습니다.


## 검색 결과를 답변 컨텍스트로 사용

검색된 메모리는 자동으로 모델에 들어가지 않습니다. 애플리케이션이 검색 정책과
프롬프트 삽입 방식을 명시해야 합니다.


In [6]:
from langchain.chat_models import init_chat_model

MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.4-mini")
model = init_chat_model(MODEL_ID)

query = "면접자가 프로젝트에서 맡은 역할을 한 문장으로 답해 주세요."
memories = memory_store.search(namespace, query=query, limit=2)
context = "\n\n".join(item.value["text"] for item in memories)

response = model.invoke(
    [
        {
            "role": "system",
            "content": (
                "아래 면접 기록만 근거로 답하세요. 근거가 없으면 모른다고 하세요.\n\n"
                f"면접 기록:\n{context}"
            ),
        },
        {"role": "user", "content": query},
    ]
)
print(response.content)


백엔드 개발자로서 서버 로직, REST API, 데이터베이스 설계를 담당했습니다.
